# Chapter 18: Bistability resulting from rebound firing — Brian2

Representative simulations adapted from [`python/chapter18.ipynb`](../python/chapter18.ipynb) and its [chapter notes](../python/chapter18.md).

Coverage: classical HH bistability at I=8, comparing an equilibrium state with a +5 mV perturbation using identical initial gates, as in HH_BISTABLE. An algebraic root finder locates the resting state, then Brian2 integrates both neurons without voltage reset. Erisir, slow h/M-current rebound families and the long f–I continuation remain in the Python reference.

Both trajectories receive the same drive and differ only in their initial voltage. Distinct resting and repeatedly spiking outcomes at fixed drive demonstrate bistability.

Run `uv sync --group dev` in the repository root and select that environment's Python kernel. These examples require the source checkout containing `mnd/brian`; an older PyPI installation does not contain these new helpers. Run all cells in order. Brian2 performs every trajectory integration. Numerical states use the book's units; helper clocks map a source time unit to one Brian millisecond. FHN and normal-form time is abstract model time.

In [ ]:
from pathlib import Path
import sys

# Use this source checkout so newly added helpers are available.
root = next((p for p in (Path.cwd(), Path.cwd().parent) if (p / "mnd").is_dir()), None)
if root is not None:
    sys.path.insert(0, str(root))
import brian2 as b2
import numpy as np
import matplotlib.pyplot as plt
b2.prefs.codegen.target = "numpy"

In [ ]:
from scipy.optimize import brentq
from mnd.brian.models import make_neurons

def simulate_hh_bistable(current=8,duration=40,dt=.01):
    neurons=make_neurons("hh",n=2,current=current,dt=dt)
    def balance(v):
        neurons.v=v
        m=float(neurons.m_inf[0]);h=float(neurons.h_inf[0]);n=float(neurons.n_inf[0])
        return 120*m**3*h*(45-v)+36*n**4*(-82-v)+.3*(-59-v)+current
    rest=brentq(balance,-75,-50)
    neurons.v=rest
    neurons.m="m_inf";neurons.h="h_inf";neurons.n="n_inf"
    neurons.v[1]=rest+5
    trace=b2.StateMonitor(neurons,"v",record=True)
    spikes=b2.SpikeMonitor(neurons)
    network=b2.Network(neurons,trace,spikes)
    network.run(duration*b2.ms,namespace={})
    return trace,spikes,rest

trace,spikes,rest=simulate_hh_bistable()
plt.figure(figsize=(9,4))
for i,label in enumerate(["resting equilibrium","+5 mV perturbation"]):
    plt.plot(trace.t/b2.ms,trace.v[i],label=label)
plt.xlabel("t (ms)");plt.ylabel("v (mV)");plt.legend();plt.tight_layout();plt.show()
assert np.ptp(trace.v[0])<1e-6
assert spikes.count[1]>=2
print(f"Resting voltage: {rest:.6f} mV; spike counts: {np.asarray(spikes.count)}")